In [ ]:
# Download the data
import os
from dotenv import load_dotenv
from roboflow import Roboflow

load_dotenv()
api_key = os.environ.get("ROBOFLOW_API_KEY")

rf = Roboflow(api_key=api_key)
project = rf.workspace("cybertech-qde01").project("waste-classification-q75av-awlnx")
version = project.version(1)
dataset = version.download("multiclass")

In [ ]:
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

base_path = dataset.location

# REMOVED "trash" from the explicit class list.
# We will detect trash by "low confidence" in these 5.
class_names = ["cardboard", "glass", "metal", "paper", "plastic"]

def load_roboflow_split(split_name, batch_size=32, image_size=(224, 224)):
    folder_path = os.path.join(base_path, split_name)
    csv_path = os.path.join(folder_path, "_classes.csv")
        
    df = pd.read_csv(csv_path)

    # Filter out "trash" images from training!
    # "Trash" images have 0 entries for all the specific classes.
    # We only want to train on images that actually belong to one of our 5 classes.
    if split_name == "train":
        # Keep rows where at least one of the class columns is 1
        df = df[df[class_names].sum(axis=1) > 0]
    
    # Normalize image data
    datagen = ImageDataGenerator(rescale=1./255)
    
    generator = datagen.flow_from_dataframe(
        dataframe=df,
        directory=folder_path,
        x_col="filename",
        y_col=class_names, 
        target_size=image_size,
        batch_size=batch_size,
        class_mode="raw", 
        shuffle=(split_name == "train") 
    )
    
    return generator

train_gen = load_roboflow_split("train", batch_size=64)
valid_gen = load_roboflow_split("valid", batch_size=64)
test_gen = load_roboflow_split("test", batch_size=64)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_image(images, labels=None, predictions=None, cmap=None, n_preds=10):
    show_label = True
    if labels is None:
        labels = images
        show_label = False

    show_prediction = True
    if predictions is None:
        predictions = images
        show_prediction = False

    n_preds = min(n_preds, len(images))

    for image, label, prediction in zip(images[:n_preds], labels[:n_preds], predictions[:n_preds]):
        plt.xticks([])
        plt.yticks([])
        plt.imshow(image, cmap=cmap)

        label_text = ""

        if show_prediction:
          cls_pred = tf.math.argmax(prediction, axis=0)
          class_number = class_names[cls_pred] 

          label_text = f"\npredicted: {class_names[cls_pred]}, probability: {round(prediction[cls_pred], 2)}"

        if show_label:
            class_idx = np.argmax(label)
            plt.xlabel(f"{class_names[class_idx]}" + label_text)

        plt.show()

    if show_prediction:
      matches = tf.keras.metrics.categorical_accuracy(labels[:n_preds], predictions[:n_preds])
      print("accuracy on this plotted sample: ", tf.reduce_mean(matches).numpy())

In [ ]:

filters = 2
X_batch, y_batch = next(iter(train_gen))
sample_image, sample_label = X_batch[:1], y_batch[:1]
plot_image(sample_image, sample_label)

sample_convolution = tf.keras.layers.Conv2D(
    filters=filters,
    kernel_size=(3,3),
    activation='relu', # !!!!!
    padding="same",
    use_bias=False
)

sample_transformed = sample_convolution(sample_image)

sample_kernel = tf.constant([
    [1,      1,     1],
    [0,      0,     0],
    [-1,    -1,    -1]
])

# Stacking the same kernel for three R/G/B channels
sample_kernel = tf.stack([sample_kernel, sample_kernel, sample_kernel], axis=-1)

# Stacking the kernel for both filters (filters=2)
sample_kernel_filters = tf.stack([sample_kernel, sample_kernel], axis=-1)

sample_convolution.set_weights([sample_kernel_filters])

sample_transformed = sample_convolution(sample_image)

plot_image(sample_image, sample_label)

for image in range(filters):
    # ... = :, :, :
    plot_image(sample_transformed[..., image])

In [ ]:
model = tf.keras.Sequential([
    # input = generator's output size (224, 224, 3)
    tf.keras.Input(shape=(224, 224, 3)),

    # Conv2D - Slides the kernel
    # First arg (32) is the amount of filters to learn (the numbers in the kernel)
    # Early layers should have fewer filters because they learn simpler shapes (eg. vertical lines)
    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),

    # MaxPooling2D - Shrinks image size by picking max number in a given pool ("window"), removing noise
    # default pool size is (2,2) which means it's shrinking the image by half in width and height
    tf.keras.layers.MaxPooling2D(), 
    
    # Learning medium complexity shapes
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    # Learning advanced shapes
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(64, activation="relu"),
    
    # Output is now 5, not 6. (Because we removed trash)
    tf.keras.layers.Dense(5, activation="softmax")
])

model.summary()

In [ ]:

loss = tf.keras.losses.CategoricalCrossentropy()

metrics = [tf.keras.metrics.CategoricalAccuracy()]
callbacks = [tf.keras.callbacks.EarlyStopping(patience=3)]

model.compile(
    loss=loss,
    optimizer='adamw',
    metrics=metrics
)

model.fit(
    train_gen, 
    epochs=50, 
    validation_data=valid_gen,
    callbacks=callbacks
)

In [ ]:

X_test_batch, y_test_batch = next(iter(test_gen))
test_preds = model.predict(X_test_batch)
plot_image(X_test_batch, y_test_batch, test_preds)

In [82]:
from sklearn.metrics import classification_report
import numpy as np

def extract_average_f1(report, extra_f1=[]):
    f1_scores = []
    f1_scores.extend(extra_f1)
    for cls in class_names:
        if cls in report:
            f1_scores.append(report[cls]['f1-score'])
            
    return np.mean(f1_scores) if f1_scores else 0

def predict_from_generator(generator):
    generator.reset()
    
    probabilities = model.predict(generator)
    predicted_classes = np.argmax(probabilities, axis=1)
    
    y_true_vectors = generator.labels
    
    y_true = np.argmax(y_true_vectors, axis=1)
    
    report_string = classification_report(y_true, predicted_classes, target_names=class_names, zero_division=0)
    print(report_string)
    
    return classification_report(y_true, predicted_classes, target_names=class_names, output_dict=True, zero_division=0)

print("\n--- TEST REPORT IGNORING TRASH ---:")
report_test = predict_from_generator(test_gen)

print("\n--- VALIDATION REPORT IGNORING TRASH ---:")
report_valid = predict_from_generator(valid_gen)

# We throw in an extra 0 during calculation since in this sample trash always has an f1 score of 0
avg_f1 = np.mean([extract_average_f1(report_test, [0]), extract_average_f1(report_valid, [0])])
print(f"Average f1 across test/valid: {avg_f1:.4f}")


--- TEST REPORT IGNORING TRASH ---:
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 137ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 137ms/step
              precision    recall  f1-score   support

   cardboard       0.77      0.57      0.65        58
       glass       0.38      0.70      0.50        47
       metal       0.45      0.23      0.30        40
       paper       0.59      0.72      0.65        64
     plastic       0.77      0.45      0.57        44

    accuracy                           0.56       253
   macro avg       0.59      0.53      0.53       253
weighted avg       0.60      0.56      0.55       253


--- VALIDATION REPORT IGNORING TRASH ---:
              precision    recall  f1-score   support

   cardboard       0.77      0.57      0.65        58
       glass       0.38      0.70      0.50        47
       metal       0.45      0.23      0.30        40
       paper       0.59      0.72      0.65        64
     plastic       0.77      0.45      0.57        44

    accuracy               

In [83]:
import numpy as np
from sklearn.metrics import classification_report

def predict_with_threshold(generator, threshold=0.5):
    generator.reset()
    
    raw_probs = model.predict(generator)
    pred_classes = np.argmax(raw_probs, axis=1)
    confidence = np.max(raw_probs, axis=1)
    
    TRASH_INDEX = 5
    low_confidence_mask = confidence < threshold
    pred_classes[low_confidence_mask] = TRASH_INDEX
    
    print(f"Rejected {np.sum(low_confidence_mask)} low-confidence samples as Trash.")

    y_true = np.argmax(generator.labels, axis=1)
    is_trash = np.sum(generator.labels, axis=1) == 0
    y_true[is_trash] = TRASH_INDEX
    
    report_classes = class_names + ["trash"]
    
    print(classification_report(y_true, pred_classes, target_names=report_classes, zero_division=0))
    
    return classification_report(y_true, pred_classes, target_names=report_classes, output_dict=True, zero_division=0)

# (I calculated the optimal threshold via a for loop)
THRESHOLD = 0.35000000000000003

print("\n--- TEST DATA WITH THRESHOLD ---")
report_test_thresh = predict_with_threshold(test_gen, threshold=THRESHOLD)

print("\n--- VALIDATION DATA WITH THRESHOLD ---")
report_valid_thresh = predict_with_threshold(valid_gen, threshold=THRESHOLD)

avg_f1 = np.mean([
    extract_average_f1(report_test_thresh), 
    extract_average_f1(report_valid_thresh)
])
print(f"\nAverage f1 across test/valid: {avg_f1:.4f}")


--- TEST DATA WITH THRESHOLD ---
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 139ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 139ms/step
Rejected 9 low-confidence samples as Trash.
              precision    recall  f1-score   support

   cardboard       0.72      0.70      0.71        44
       glass       0.39      0.70      0.50        47
       metal       0.41      0.17      0.25        40
       paper       0.62      0.72      0.67        64
     plastic       0.77      0.45      0.57        44
       trash       0.11      0.07      0.09        14

    accuracy                           0.55       253
   macro avg       0.50      0.47      0.46       253
weighted avg       0.56      0.55      0.53       253


--- VALIDATION DATA WITH THRESHOLD ---
Rejected 9 low-confidence samples as Trash.
              precision    recall  f1-score   support

   cardboard       0.72      0.70      0.71        44
       glass       0.39      0.70      0.50        47
       metal       0.41      0.17      0.25        40
